# Clean Trip Data

In [1]:
import pandas as pd
import numpy as np

In [2]:
import glob

files = glob.glob("../data/raw/*citibike*.csv")

trips_list = [pd.read_csv(file) for file in files]

trips = pd.concat(trips_list, ignore_index=True)

print("Number of files:", len(files))
print("Trips shape:", trips.shape)

C:\Users\MY PC\AppData\Local\Temp\ipykernel_8704\1651997543.py:5: DtypeWarning: Columns (0: end_station_id) have mixed types. Specify dtype option on import or set low_memory=False.
  trips_list = [pd.read_csv(file) for file in files]


Number of files: 6
Trips shape: (5239373, 13)


# check duplicates values

In [5]:
trips.duplicated().sum()

np.int64(0)

# check null values

In [6]:
trips.isna().sum()

ride_id                   0
rideable_type             0
started_at                0
ended_at                  0
start_station_name     2815
start_station_id       2815
end_station_name      13922
end_station_id        14916
start_lat              2815
start_lng              2815
end_lat               14886
end_lng               14886
member_casual             0
dtype: int64

# Calculate trip duration in minutes

In [7]:
trips['trip_duration_min'] = (trips['ended_at'] - trips['started_at']).dt.total_seconds() / 60

# Remove invalid trips

In [8]:
trips = trips[(trips['trip_duration_min'] > 1) & (trips['trip_duration_min'] < 24*60)]

# Remove duplicates

In [9]:
trips = trips.drop_duplicates(subset=['ride_id'])

In [10]:
print("After cleaning trips:", trips.shape)

After cleaning trips: (5238215, 14)


In [11]:
start_missing = trips['start_station_id'].isna()
end_missing = trips['end_station_id'].isna()

print("Missing start station ID:", start_missing.sum())
print("Missing end station ID:", end_missing.sum())

print("Missing both:", (start_missing & end_missing).sum())
print("Missing start only:", (start_missing & ~end_missing).sum())
print("Missing end only:", (~start_missing & end_missing).sum())

Missing start station ID: 2815
Missing end station ID: 13790
Missing both: 362
Missing start only: 2453
Missing end only: 13428


In [15]:
print("Zero duration:", (trips['trip_duration_min'] == 0).sum())

print("Negative duration:", (trips['trip_duration_min'] < 0).sum())

print("Trips >= 24 hours:", (trips['trip_duration_min'] >= 24 * 60).sum())

Zero duration: 0
Negative duration: 0
Trips >= 24 hours: 0


In [16]:
print(trips['trip_duration_min'].describe())

count    5.238215e+06
mean     1.358837e+01
std      1.920745e+01
min      1.000083e+00
25%      5.632383e+00
50%      9.661833e+00
75%      1.664915e+01
max      1.438808e+03
Name: trip_duration_min, dtype: float64


In [17]:
trips = trips[(trips['trip_duration_min'] > 1) &
              (trips['trip_duration_min'] < 24*60)]

# Load station information

In [18]:
stations = pd.read_csv("../data/raw/station_information.csv")

print("Stations shape:", stations.shape)

Stations shape: (2520, 15)


# Clean Station Data

In [19]:
# Keep only useful columns

In [20]:
stations = stations[['station_id', 'name', 'lat', 'lon', 'capacity']].copy()

# Rename columns

In [21]:
stations = stations.rename(columns={
'name': 'station_name',
'lat': 'latitude',
'lon': 'longitude'
})

In [22]:
stations.head()

,station_id,station_name,latitude,longitude,capacity
0,bfe5e69d-89ff-48b3-bf20-80cbdfa443e0,India St Pier,40.731735,-73.961239,54
1,1851674198082642572,Corona Ave & 111 St,40.740800,-73.849780,22
2,2206779090602992084,68 Dr & 112 St,40.727040,-73.841280,12
3,4e9a1c3a-20e2-4fd5-b8de-fbf8d43e7bcc,Maple St & Flatbush Ave,40.660260,-73.960450,24
4,b4b5824b-3c23-44b7-92a3-cec6a54583a0,Morris Ave & E 171 St,40.838990,-73.910900,21


# Merge Trips + Station Information

In [23]:
trips = trips.merge(
stations,
left_on='start_station_id',
right_on='station_id',
how='left'
)

trips = trips.rename(columns={
'station_name': 'start_station_name_clean',
'latitude': 'start_lat_station',
'longitude': 'start_lng_station',
'capacity': 'start_capacity'
})
trips = trips.drop(columns=['station_id'])

In [24]:
trips.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,trip_duration_min,start_station_name_clean,start_lat_station,start_lng_station,start_capacity
0,19AE6A4FF06F412F,electric_bike,2026-07-21 20:41:12.464,2026-07-21 20:50:09.456,W 36 St & 7 Ave,6483.06,Lexington Ave & E 24 St,6089.07,40.752149,-73.989539,40.740259,-73.984092,member,8.949867,NaN,NaN,NaN,NaN
1,2FFEFDA0732D107C,electric_bike,2026-07-20 18:29:27.483,2026-07-20 18:59:39.567,Sterling Pl & Bedford Ave,3993.03,Waterbury St & Stagg St,5099.02,40.672695,-73.954131,40.710190,-73.937340,member,30.201400,NaN,NaN,NaN,NaN
2,27071C9A36571492,electric_bike,2026-07-21 15:31:35.913,2026-07-21 15:35:36.169,Emerson Pl & Myrtle Ave,4683.02,Washington Park,4546.05,40.693631,-73.962236,40.691782,-73.973730,member,4.004267,NaN,NaN,NaN,NaN
3,DB12F248E04AA93A,classic_bike,2026-07-26 22:11:59.009,2026-07-26 22:13:40.687,65 St & 35 Ave,6417.04,37 Ave & 63 St,6340.09,40.750530,-73.897700,40.748930,-73.899810,member,1.694633,NaN,NaN,NaN,NaN
4,B21338894CD26018,electric_bike,2026-07-17 21:19:10.032,2026-07-17 21:28:17.408,E 55 St & 2 Ave,6650.07,W 54 St & 9 Ave,6920.05,40.757973,-73.966033,40.766040,-73.987370,member,9.122933,NaN,NaN,NaN,NaN


In [25]:
trips = trips.merge(
stations,
left_on='end_station_id',
right_on='station_id',
how='left'
)

In [26]:
trips = trips.rename(columns={
'station_name': 'end_station_name_clean',
'latitude': 'end_lat_station',
'longitude': 'end_lng_station',
'capacity': 'end_capacity'
})
trips = trips.drop(columns=['station_id'])

In [27]:
trips.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,...,member_casual,trip_duration_min,start_station_name_clean,start_lat_station,start_lng_station,start_capacity,end_station_name_clean,end_lat_station,end_lng_station,end_capacity
0,19AE6A4FF06F412F,electric_bike,2026-07-21 20:41:12.464,2026-07-21 20:50:09.456,W 36 St & 7 Ave,6483.06,Lexington Ave & E 24 St,6089.07,40.752149,-73.989539,...,member,8.949867,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2FFEFDA0732D107C,electric_bike,2026-07-20 18:29:27.483,2026-07-20 18:59:39.567,Sterling Pl & Bedford Ave,3993.03,Waterbury St & Stagg St,5099.02,40.672695,-73.954131,...,member,30.201400,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,27071C9A36571492,electric_bike,2026-07-21 15:31:35.913,2026-07-21 15:35:36.169,Emerson Pl & Myrtle Ave,4683.02,Washington Park,4546.05,40.693631,-73.962236,...,member,4.004267,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,DB12F248E04AA93A,classic_bike,2026-07-26 22:11:59.009,2026-07-26 22:13:40.687,65 St & 35 Ave,6417.04,37 Ave & 63 St,6340.09,40.750530,-73.897700,...,member,1.694633,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,B21338894CD26018,electric_bike,2026-07-17 21:19:10.032,2026-07-17 21:28:17.408,E 55 St & 2 Ave,6650.07,W 54 St & 9 Ave,6920.05,40.757973,-73.966033,...,member,9.122933,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Load weather

In [28]:
weather = pd.read_csv("../data/raw/nyc_weather.csv")

print("Weather shape:", weather.shape)

Weather shape: (2208, 5)


In [45]:
print(weather["time"].min())
print(weather["time"].max())
print(weather.columns.tolist())

2026-07-14 00:00:00
2026-08-31 23:00:00
['time', 'temperature_2m', 'relative_humidity_2m', 'precipitation', 'wind_speed_10m', 'date_hour']


# Merge with Weather Data

In [46]:
weather["time"] = pd.to_datetime(weather["time"])

weather["date_hour"] = weather["time"].dt.floor("h")
trips["date_hour"] = trips["started_at"].dt.floor("h")

trips = trips.merge(
    weather,
    on="date_hour",
    how="left"
)

print("After weather merge:", trips.shape)

After weather merge: (5238215, 33)


In [52]:
print(
    trips[
        ["temperature_2m_x", "temperature_2m_y"]
    ].head()
)

   temperature_2m_x  temperature_2m_y
0              24.9              24.9
1              27.2              27.2
2              22.9              22.9
3              23.8              23.8
4              26.8              26.8


In [53]:
weather_cols = [
    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "wind_speed_10m"
]

for col in weather_cols:
    trips[col] = trips[col + "_y"]

trips = trips.drop(
    columns=[
        "temperature_2m_x", "temperature_2m_y",
        "relative_humidity_2m_x", "relative_humidity_2m_y",
        "precipitation_x", "precipitation_y",
        "wind_speed_10m_x", "wind_speed_10m_y"
    ]
)

print(trips.shape)
print(trips[weather_cols].isnull().sum())

(5238215, 29)
temperature_2m          0
relative_humidity_2m    0
precipitation           0
wind_speed_10m          0
dtype: int64


In [56]:
trips.to_csv(
    "../data/processed/citibike_trips_merged.csv",
    index=False
)

print("Merged dataset saved successfully!")

Merged dataset saved successfully!


In [57]:
import pandas as pd

df = pd.read_csv("../data/processed/citibike_trips_merged.csv")

C:\Users\MY PC\AppData\Local\Temp\ipykernel_8704\1899333067.py:3: DtypeWarning: Columns (0: end_station_id) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/processed/citibike_trips_merged.csv")


In [58]:
df.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,...,end_lat_station,end_lng_station,end_capacity,date_hour,time_x,time_y,temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m
0,19AE6A4FF06F412F,electric_bike,2026-07-21 20:41:12.464,2026-07-21 20:50:09.456,W 36 St & 7 Ave,6483.06,Lexington Ave & E 24 St,6089.07,40.752149,-73.989539,...,NaN,NaN,NaN,2026-07-21 20:00:00,2026-07-21 20:00:00,2026-07-21 20:00:00,24.9,89,0.5,13.1
1,2FFEFDA0732D107C,electric_bike,2026-07-20 18:29:27.483,2026-07-20 18:59:39.567,Sterling Pl & Bedford Ave,3993.03,Waterbury St & Stagg St,5099.02,40.672695,-73.954131,...,NaN,NaN,NaN,2026-07-20 18:00:00,2026-07-20 18:00:00,2026-07-20 18:00:00,27.2,35,0.0,12.2
2,27071C9A36571492,electric_bike,2026-07-21 15:31:35.913,2026-07-21 15:35:36.169,Emerson Pl & Myrtle Ave,4683.02,Washington Park,4546.05,40.693631,-73.962236,...,NaN,NaN,NaN,2026-07-21 15:00:00,2026-07-21 15:00:00,2026-07-21 15:00:00,22.9,93,2.7,16.7
3,DB12F248E04AA93A,classic_bike,2026-07-26 22:11:59.009,2026-07-26 22:13:40.687,65 St & 35 Ave,6417.04,37 Ave & 63 St,6340.09,40.750530,-73.897700,...,NaN,NaN,NaN,2026-07-26 22:00:00,2026-07-26 22:00:00,2026-07-26 22:00:00,23.8,61,0.0,9.0
4,B21338894CD26018,electric_bike,2026-07-17 21:19:10.032,2026-07-17 21:28:17.408,E 55 St & 2 Ave,6650.07,W 54 St & 9 Ave,6920.05,40.757973,-73.966033,...,NaN,NaN,NaN,2026-07-17 21:00:00,2026-07-17 21:00:00,2026-07-17 21:00:00,26.8,44,0.0,7.3


# convert into datetime

In [59]:
df["started_at"] = pd.to_datetime(df["started_at"])
df["ended_at"] = pd.to_datetime(df["ended_at"])

print(df["started_at"].dtype)
print(df["ended_at"].dtype)

datetime64[us]
datetime64[us]


In [60]:
df["hour"] = df["started_at"].dt.hour
df["day_of_week"] = df["started_at"].dt.day_name()
df["is_weekend"] = df["started_at"].dt.dayofweek >= 5

print(df[[
    "started_at",
    "hour",
    "day_of_week",
    "is_weekend"
]].head())

               started_at  hour day_of_week  is_weekend
0 2026-07-21 20:41:12.464    20     Tuesday       False
1 2026-07-20 18:29:27.483    18      Monday       False
2 2026-07-21 15:31:35.913    15     Tuesday       False
3 2026-07-26 22:11:59.009    22      Sunday        True
4 2026-07-17 21:19:10.032    21      Friday       False


# We use the Haversine formula because we want to calculate the approximate distance between the starting station and ending station of each bike trip.

In [61]:
import numpy as np

def haversine(lat1, lon1, lat2, lon2):
    R = 6371

    lat1, lon1, lat2, lon2 = map(
        np.radians,
        [lat1, lon1, lat2, lon2]
    )

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin(dlon / 2) ** 2
    )

    return 2 * R * np.arcsin(np.sqrt(a))

In [62]:
df["trip_distance_km"] = haversine(
    df["start_lat"],
    df["start_lng"],
    df["end_lat"],
    df["end_lng"]
)

print(df[[
    "start_lat",
    "start_lng",
    "end_lat",
    "end_lng",
    "trip_distance_km"
]].head())

   start_lat  start_lng    end_lat    end_lng  trip_distance_km
0  40.752149 -73.989539  40.740259 -73.984092          1.399493
1  40.672695 -73.954131  40.710190 -73.937340          4.403046
2  40.693631 -73.962236  40.691782 -73.973730          0.990657
3  40.750530 -73.897700  40.748930 -73.899810          0.251485
4  40.757973 -73.966033  40.766040 -73.987370          2.008467


# check Trip_distance statistics

In [63]:
df["trip_distance_km"].describe()

count    5.221974e+06
mean     2.122737e+00
std      1.804879e+00
min      0.000000e+00
25%      9.033213e-01
50%      1.590772e+00
75%      2.782932e+00
max      3.523348e+01
Name: trip_distance_km, dtype: float64

In [64]:
df["trip_distance_km"].isnull().sum()

np.int64(16241)

In [65]:
df.columns.tolist()

['ride_id',
 'rideable_type',
 'started_at',
 'ended_at',
 'start_station_name',
 'start_station_id',
 'end_station_name',
 'end_station_id',
 'start_lat',
 'start_lng',
 'end_lat',
 'end_lng',
 'member_casual',
 'trip_duration_min',
 'start_station_name_clean',
 'start_lat_station',
 'start_lng_station',
 'start_capacity',
 'end_station_name_clean',
 'end_lat_station',
 'end_lng_station',
 'end_capacity',
 'date_hour',
 'time_x',
 'time_y',
 'temperature_2m',
 'relative_humidity_2m',
 'precipitation',
 'wind_speed_10m',
 'hour',
 'day_of_week',
 'is_weekend',
 'trip_distance_km']

# select only our final columns

In [66]:
final_columns = [
    "ride_id",
    "rideable_type",
    "started_at",
    "ended_at",
    "trip_duration_min",
    "trip_distance_km",
    "member_casual",

    "start_station_id",
    "start_station_name",
    "start_lat",
    "start_lng",
    "start_capacity",

    "end_station_id",
    "end_station_name",
    "end_lat",
    "end_lng",
    "end_capacity",

    "hour",
    "day_of_week",
    "is_weekend",

    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "wind_speed_10m"
]

df = df[final_columns]

print("Final shape:", df.shape)
print(df.columns.tolist())

Final shape: (5238215, 24)
['ride_id', 'rideable_type', 'started_at', 'ended_at', 'trip_duration_min', 'trip_distance_km', 'member_casual', 'start_station_id', 'start_station_name', 'start_lat', 'start_lng', 'start_capacity', 'end_station_id', 'end_station_name', 'end_lat', 'end_lng', 'end_capacity', 'hour', 'day_of_week', 'is_weekend', 'temperature_2m', 'relative_humidity_2m', 'precipitation', 'wind_speed_10m']


In [68]:
df.to_csv(
    "../data/processed/citibike_final_cleaned.csv",
    index=False
)

print("Final cleaned dataset saved successfully!")

Final cleaned dataset saved successfully!
